# Prática de Laboratório 10: OpenMP Tasks

**Tema:** criação, execução e sincronização básica de tarefas em OpenMP.  
**Escopo:** esta prática **não** utiliza dependência entre tarefas (`depend`).  
**Ambiente sugerido:** Google Colab, usando células no padrão Jupyter Notebook.

## Objetivos

Ao final da prática, você deverá ser capaz de:

1. Diferenciar tarefas implícitas e explícitas em OpenMP.
2. Usar corretamente `parallel`, `single`, `task` e `taskwait`.
3. Observar que a ordem de execução das tarefas não é determinada pelo programa.
4. Identificar situações em que tarefas são criadas por uma thread e executadas por outra.
5. Corrigir problemas simples de escopo de variáveis com `firstprivate` e `shared`.
6. Aplicar tarefas em algoritmos recursivos simples, como Fibonacci e Quicksort.
7. Avaliar o efeito da granularidade das tarefas usando uma estratégia de corte com `if`.

## Orientações gerais

Execute as células na ordem em que aparecem. Em algumas atividades, será necessário executar o mesmo programa mais de uma vez e comparar as saídas. Em outras, haverá trechos marcados com `TODO`, que devem ser modificados por você.

Ao final de cada atividade, responda às perguntas propostas em uma célula de texto.


## 1. Criando tarefas com `task`

Agora será usada a diretiva `#pragma omp task`. Observe também o uso de `single`: apenas uma thread cria as tarefas, mas as tarefas podem ser executadas por qualquer thread da equipe.

### Código

In [ ]:

%%writefile exemplo_tasks.c
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel num_threads(8)
    #pragma omp single
    {
        printf("A ");

        #pragma omp task
        {
         usleep(100);
         printf("race ");
        }

        #pragma omp task
        printf("car ");
    }

    printf("\n");
    return 0;
}


Writing exemplo_tasks.c


In [ ]:

!gcc -fopenmp exemplo_tasks.c -o exemplo_tasks
!echo "Execução 1:" && ./exemplo_tasks
!echo "Execução 2:" && ./exemplo_tasks
!echo "Execução 3:" && ./exemplo_tasks


Execução 1:
A car race 
Execução 2:
A race car 
Execução 3:
A car race 


### Perguntas

1. Por que a palavra `A` aparece apenas uma vez?
2. As palavras `race` e `car` aparecem sempre na mesma ordem?
3. O que poderia acontecer se a diretiva `single` fosse removida?
4. Em que momento as tarefas pendentes têm garantia de terminar neste programa?


## 2. Sincronização básica com `taskwait`

A diretiva `taskwait` força a tarefa atual a esperar pelo término de suas tarefas-filhas diretas.

Nesta atividade, compare duas versões: uma sem `taskwait` e outra com `taskwait`.

### Código


In [ ]:

%%writefile exemplo_taskwait.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    int usar_taskwait = 0;

    if (argc > 1) {
        usar_taskwait = atoi(argv[1]);
    }

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        printf("A ");

        #pragma omp task
        printf("race ");

        #pragma omp task
        printf("car ");

        if (usar_taskwait) {
            #pragma omp taskwait
        }

        printf("is fun to watch ");
    }

    printf("\n");
    return 0;
}


Overwriting exemplo_taskwait.c


In [ ]:

!gcc -fopenmp exemplo_taskwait.c -o exemplo_taskwait

!echo "Sem taskwait:"
!./exemplo_taskwait 0
!./exemplo_taskwait 0
!./exemplo_taskwait 0

!echo ""
!echo "Com taskwait:"
!./exemplo_taskwait 1
!./exemplo_taskwait 1
!./exemplo_taskwait 1


### Perguntas

1. Sem `taskwait`, a frase `is fun to watch` pode aparecer antes de `race` e `car`?
2. Com `taskwait`, o que muda?
3. O `taskwait` impõe ordem entre `race` e `car`? Justifique.
4. O `taskwait` espera todas as tarefas do programa ou apenas as tarefas-filhas diretas da tarefa atual?


## 3. Thread que cria tarefa × thread que executa tarefa

O ambiente de execução do OpenMP decide quando e por qual thread cada tarefa será executada. Uma thread pode encontrar/criar uma tarefa, mas outra thread pode executá-la.

### Código


In [ ]:

%%writefile exemplo_criador_executor.c
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <omp.h>

int main(int argc, char **argv) {
    int ntarefas = 12;

    if (argc > 1) {
        ntarefas = atoi(argv[1]);
    }

    #pragma omp parallel num_threads(4)
    #pragma omp single
    {
        int criador = omp_get_thread_num();

        for (int i = 0; i < ntarefas; i++) {
            #pragma omp task firstprivate(i, criador)
            {
                usleep(1000 * (10 + (i % 5) * 20));
                printf("Tarefa %2d: criada pela thread %d, executada pela thread %d\n",
                       i, criador, omp_get_thread_num());
            }
        }

        #pragma omp taskwait
        printf("Todas as tarefas-filhas da região single terminaram.\n");
    }

    return 0;
}


Writing exemplo_criador_executor.c


In [ ]:

!gcc -fopenmp exemplo_criador_executor.c -o exemplo_criador_executor
!./exemplo_criador_executor 16


### Perguntas

1. Qual thread criou as tarefas?
2. Todas as tarefas foram executadas pela thread que as criou?
3. O que este exemplo mostra sobre o pool de tarefas?
4. Execute o programa com `num_threads=2`, `4` e `8`. O comportamento é exatamente igual?


### 4. Escopo de variáveis em tarefas

Nesta atividade, utilizamos uma versão expandida do exemplo apresentado em sala de aula para verificar o comportamento do escopo das variáveis com uso de tasks.

### Código


In [ ]:

%%writefile exemplo_escopo.c
#include <stdio.h>
#include <omp.h>

int a = 1;              // variável global: shared
static int g = 10;      // variável global estática: shared

void foo(void) {
    int b = 2, c = 3;   // b será private na região parallel; c será shared

    #pragma omp parallel private(b) num_threads(4)
    {
        int d = 40;
        static int s_parallel = 1000;

        #pragma omp single
        {
          #pragma omp task
          {
            int f = 5;
            static int s_task = 2000;

                printf("a = %d \n", a);

                printf("g = %d \n", g);

                printf("b = %d \n", b);

                printf("c = %d \n", c);

                printf("d = %d \n", d);

                printf("f = %d \n", f);

                printf("s_parallel = %d \n", s_parallel);

                printf("s_task = %d \n", s_task);

                printf("\n");
          }
        }
    }
}

int main(void) {
    foo();
    return 0;
}


Overwriting exemplo_firstprivate.c


In [ ]:

!gcc -fopenmp exemplo_escopo.c -o exemplo_escopo
!./exemplo_escopo


### Perguntas

1. Justifique os resultados impressos indicando para cada variável o seu escopo como:
  - shared (global)
  - shared (static)
  - shared (função)
  - private
  - firstprivate

2. Utilize a cláusula default (none) e atribua explicitamente o escopo das variáveis de modo que o resultado impresso seja o mesmo que o código original.


## 5. Fibonacci com tasks

A sequência de Fibonacci é definida por:

$
F_0 = 0,\quad F_1 = 1,\quad F_n = F_{n-1} + F_{n-2}
$

A recursão de Fibonacci é um exemplo didático para introduzir tasks, mas também mostra um problema importante: tarefas muito pequenas podem gerar overhead maior que o ganho de paralelismo.

Nesta atividade, usaremos a cláusula `if` para controlar a granularidade das tarefas.

### Código


In [ ]:

%%writefile exemplo_fibonacci_tasks.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

long fib_seq(int n) {
    if (n < 2) {
        return n;
    }

    return fib_seq(n - 1) + fib_seq(n - 2);
}

long fib_task(int n, int cutoff) {
    if (n < 2) {
        return n;
    }

    long x, y;

    #pragma omp task shared(x)  firstprivate(n, cutoff) if(n > cutoff)
    {
        x = fib_task(n - 1, cutoff);
    }

    #pragma omp task shared(y) firstprivate(n, cutoff) if(n > cutoff)
    {
        y = fib_task(n - 2, cutoff);
    }

    #pragma omp taskwait
    return x + y;
}

long executar_fib_task(int n, int cutoff, int nthreads) {
    long resultado = 0;

    #pragma omp parallel num_threads(nthreads)
    #pragma omp single
    {
        resultado = fib_task(n, cutoff);
    }

    return resultado;
}

int main(int argc, char **argv) {
    int n = 35;
    int cutoff = 20;
    int nthreads = 4;

    if (argc > 1) n = atoi(argv[1]);
    if (argc > 2) cutoff = atoi(argv[2]);
    if (argc > 3) nthreads = atoi(argv[3]);

    double t0, t1;

    t0 = omp_get_wtime();
    long rseq = fib_seq(n);
    t1 = omp_get_wtime();
    double tempo_seq = t1 - t0;

    t0 = omp_get_wtime();
    long rtask = executar_fib_task(n, cutoff, nthreads);
    t1 = omp_get_wtime();
    double tempo_task = t1 - t0;

    printf("n = %d, cutoff = %d, threads = %d\n", n, cutoff, nthreads);
    printf("fib_seq  = %ld, tempo = %.6f s\n", rseq, tempo_seq);
    printf("fib_task = %ld, tempo = %.6f s\n", rtask, tempo_task);

    if (rseq != rtask) {
        printf("ERRO: resultados diferentes!\n");
        return 1;
    }

    return 0;
}


Writing exemplo_fibonacci_tasks.c


In [ ]:

!gcc -O2 -fopenmp exemplo_fibonacci_tasks.c -o exemplo_fibonacci_tasks

!echo "Teste inicial:"
!./exemplo_fibonacci_tasks 50 30 8


Teste inicial:
n = 50, cutoff = 30, threads = 8
fib_seq  = 12586269025, tempo = 50.574271 s
fib_task = 12586269025, tempo = 922.019602 s


### Experimentos

Execute a célula seguinte e compare os tempos para diferentes valores de `cutoff`.


In [ ]:

!echo "Variação do cutoff com 6 threads:"
!for c in 10 15 20 25 30 35; do ./exemplo_fibonacci_tasks 40 $c 6; echo ""; done


In [ ]:

!echo "Variação do número de threads com cutoff fixo:"
!for t in 1 2 4 8; do ./exemplo_fibonacci_tasks 40 20 $t; echo ""; done


### Perguntas

1. Por que a versão com tasks pode ser mais lenta para alguns valores de `cutoff`?
2. Qual foi o melhor valor de `cutoff` no seu ambiente?
3. O que acontece quando `cutoff` é muito pequeno?
4. O que acontece quando `cutoff` é muito grande?
5. Por que `x` e `y` foram declarados como `shared` nas tarefas?
6. Há necessidade de `n` e `cutoff` serem declarados como `firstprivate`?
